# Creating Graphs for Complex code in TensorFlow and Machine Learning
- In tensorFlow, computation is represented as a dataflow graph, which is particularly useful for optimizing machine learning models by enabling parallel execution, distributed training, and effecient computation. When dealing with complex code in Tensorflow, managing control dependencies, loops and tracing variables is essential for ensuring correcness and performance. 

## Computational Graphs in TensorFlow
- TensorFlow computation is structured as directed graph, where:
1. Nodes represent operations (e.g, matrix multiplication, activation functions)
2. Edges represent tensors flowing between operations

- TensorFlow 1.x required explicitly defining the graph before execution, while TensorFlow 2.x uses eager execution by default, making it more intuitive. However, for perfomance optimization, tf.function allows converting Python code into a tensorflow graph

In [ ]:
import tensorflow as tf

@tf.function #converts the function into a computational graph
def add(x, y):
    return x + y

print(add(tf.constant(3), tf.constand(5))) # output: 8

# Control dependencies and control flow
- Control dependencies ensure that certain operations execute in a specific order. In TF 1.x tf.control_dependencies was used, but in TF 2.x, tf.function automatically handles dependencies. 

In [ ]:
@tf.function
def complex_operation():
    x = tf.Variable(0)

    with tf.control_dependencies([x.assign_add(1)]):  # Ensure `assign_add` runs before `print`
        y = tf.print("Updated x:", x)

    return y

complex_operation()


# Loops in TensorFlow Graphs
- TensorFlow provides tf.While_loop and tf.map_fn for loops inside graphs. 


In [ ]:
@tf.function
def loop_example(n):
    i = tf.constant(0)
    result = tf.constant(0)

    def condition(i, _):
        return i < n

    def body(i, result):
        return i + 1, result + i

    _, total = tf.while_loop(condition, body, [i, result])
    return total

print(loop_example(5))  # Output: 10 (0+1+2+3+4)


# Key points

- The loop condition and body are defined as functions. 
- tf.while_loop ensures execution within a single computation graph
- Efficiently executes on GPUs/TPUs
- tf.map_dn applies a function to each element in a Tensor, avoiding explicit Python loops. 

In [ ]:
@tf.function
def square_elements(tensor):
    return tf.map_fn(lambda x: x ** 2, tensor)

print(square_elements(tf.constant([1, 2, 3, 4])))


# Tracing Variables in TensorFlow
- Since TensorFlow 2.x uses autograph(automatically converts Python functions to graph), tracing variables becomes crucial.

## tf.function
- The function is traced only once for a given input signature 
- If inputs change type/shape, TensorFlow retraces

In [ ]:
@tf.function
def trace_example(x):
    print("Tracing function...")  # Runs only once
    return x * 2

print(trace_example(tf.constant(5)))  # Tracing occurs here
print(trace_example(tf.constant(10)))  # No re-tracing


# Inspecting Graphs
- To inspect a tensorflow graph, we use:


In [ ]:
concrete_function = trace_example.get_concrete_function(tf.constant(5))
print(concrete_function.graph.as_graph_def())


# Handling Complex Code: Best Practices
1. Use tf.function for performance
2. Minimize re-tracing by keeping consistent input shapes and types
3. Use tf.control_dependencies for execution order if needed
4. Utilize tf.while_loop and tf.map_fn for efficient looping
4. Inspect computation graphs to debug performance bottlenecks